# 🔥 FireWatch — train on a Kaggle GPU (v3: oversampling + warm-start)

Stabilized RetinaNet fine-tune on **Kaggle** (separate free GPU quota; **Save & Run All (Commit)**
runs unattended ~9-12 h). **v3 changes vs v2:** oversample the weak cases (**fire** + **indoor**),
optional **warm-start** from the current shipping model (continue, don't restart), plus the existing
augment + exposure + 512px recipe. Output: `fire_retinanet_v3.weights.h5`.

### One-time setup (right-hand panel, before running)
1. **Settings → Accelerator → GPU** (T4 or P100).
2. **Settings → Internet → On** (needed for `git clone` + `pip`).
3. **Add Data →** your private **`D-Fire-merged`** dataset (has `train/`, `test/`, `indoor_test/`).
4. *(Optional but recommended)* **Add Data →** a dataset containing the current
   **`fire_retinanet_indoor.weights.h5`** so the run can **warm-start** from it (much faster, usually
   better). If you skip this, it trains from the COCO backbone — still works, just slower to converge.

Then **Save Version → Save & Run All (Commit)**. The model lands in `/kaggle/working/` — download it
from the **Output** panel when done. **Keep v3 only if it beats overall F1 0.575 / indoor 0.666.**

## 1. Confirm the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import tensorflow as tf
print('TensorFlow', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))
assert tf.config.list_physical_devices('GPU'), 'No GPU! Settings > Accelerator > GPU, then restart.'

## 2. Get the code + install KerasHub
Needs **Internet = On**. If pip asks to restart the kernel, do it and re-run this cell.

In [ ]:
import os
os.chdir('/kaggle/working')
if not os.path.isdir('/kaggle/working/fire-detection'):
    !git clone --depth 1 -b master https://github.com/01End/fire-detection.git
os.chdir('/kaggle/working/fire-detection')
!pip install -q -U keras-hub opencv-python-headless
import keras_hub, keras
print('keras', keras.__version__, '| keras_hub', keras_hub.__version__)

## 3. Locate the D-Fire dataset (auto-detects layout under /kaggle/input)

In [ ]:
search_base = '/kaggle/input'
root = layout = None
for dp, dns, fns in os.walk(search_base):
    if os.path.isdir(os.path.join(dp, 'train', 'images')):
        root, layout = dp, 'split_first'; break
    if os.path.isdir(os.path.join(dp, 'images', 'train')):
        root, layout = dp, 'type_first'; break
assert root, f'No dataset under {search_base}. Did you Add Data (your merged D-Fire dataset)?'

if layout == 'split_first':
    VAL_SPLIT = 'test' if os.path.isdir(f'{root}/test/images') else 'val'
    DATA = root
else:  # YOLO images/<split> -> symlink into a writable <split>/images layout
    VAL_SPLIT = 'test' if os.path.isdir(f'{root}/images/test') else 'val'
    DATA = '/kaggle/working/data_norm'
    for sp in ['train', VAL_SPLIT]:
        os.makedirs(f'{DATA}/{sp}', exist_ok=True)
        for kind in ['images', 'labels']:
            dst = f'{DATA}/{sp}/{kind}'
            if not os.path.exists(dst):
                os.symlink(f'{root}/{kind}/{sp}', dst)

IMG = 512
OUT = '/kaggle/working/fire_retinanet_v3.weights.h5'

# Optional warm-start: if you added a dataset containing a .weights.h5, fine-tune from it
# instead of the COCO backbone (faster, usually better). Don't pick up our own output name.
INIT = None
for dp, dns, fns in os.walk(search_base):
    for fn in fns:
        if fn.endswith('.weights.h5') and 'v3' not in fn:
            INIT = os.path.join(dp, fn); break
    if INIT:
        break

print('DATA =', DATA, '| VAL_SPLIT =', VAL_SPLIT, '| layout =', layout)
print('train images:', len(os.listdir(f'{DATA}/train/images')),
      f'| {VAL_SPLIT}:', len(os.listdir(f'{DATA}/{VAL_SPLIT}/images')))
print('warm-start from:', INIT or '(none -> training from the COCO backbone)')

## 4. Train (stable) 🚀
Low LR (1e-4) + reduce-LR-on-plateau + early-stopping; best weights mirrored to `/kaggle/working`.
`--epochs 16` is a ceiling — early stopping ends it when val stalls. Warm-start (if a model dataset
was added) means it fine-tunes from the current model instead of COCO, so it converges faster.

In [ ]:
EXPOSURE = 'clahe'   # set to 'none' for the safer, proven config (must match eval/inference)
INIT_FLAG = f'--init-weights "{INIT}"' if INIT else ''
# Oversample the weak cases so the fine-tune sees them more: fire (hard class) and indoor (weak
# domain). Additive repeat -> indoor-fire images (rarest, most valuable) are seen the most.
!python -m training.tf_train \
    --data "{DATA}" \
    --train-split train \
    --val-split {VAL_SPLIT} \
    --epochs 16 \
    --batch-size 8 \
    --image-size {IMG} \
    --lr 0.0001 \
    --augment --exposure {EXPOSURE} \
    --oversample-fire 2 --oversample-indoor 2 \
    {INIT_FLAG} \
    --out "{OUT}"

## 5. Measure accuracy on the held-out test set 📊

In [ ]:
# --- Baseline: global 0.6 operating point (compare directly to v2 / current 0.575 / 0.666) ---
!python -m training.tf_eval --model "{OUT}" --data "{DATA}/{VAL_SPLIT}" \
    --image-size {IMG} --score-thr 0.6 --exposure {EXPOSURE} --limit 500
import os
if os.path.isdir(f'{DATA}/indoor_test/images'):
    print('\n===== INDOOR-ONLY (global 0.6) =====')
    !python -m training.tf_eval --model "{OUT}" --data "{DATA}/indoor_test" \
        --image-size {IMG} --score-thr 0.6 --exposure {EXPOSURE} --limit 750

# --- Per-class threshold sweep: best fire/smoke thresholds + the F1 gain over global 0.6 ---
print('\n===== PER-CLASS THRESHOLD SWEEP (overall test) =====')
!python -m training.tf_tune_thresholds --model "{OUT}" --data "{DATA}/{VAL_SPLIT}" \
    --image-size {IMG} --exposure {EXPOSURE} --limit 500
if os.path.isdir(f'{DATA}/indoor_test/images'):
    print('\n===== PER-CLASS THRESHOLD SWEEP (indoor only) =====')
    !python -m training.tf_tune_thresholds --model "{OUT}" --data "{DATA}/indoor_test" \
        --image-size {IMG} --exposure {EXPOSURE} --limit 750

## 6. See it work — annotated detections 🔥

In [ ]:
import glob, shutil
os.makedirs('/kaggle/working/val_sample', exist_ok=True)
picked = 0
for lf in sorted(glob.glob(f'{DATA}/{VAL_SPLIT}/labels/*.txt')):
    if os.path.getsize(lf) > 0:
        stem = os.path.splitext(os.path.basename(lf))[0]
        ip = f'{DATA}/{VAL_SPLIT}/images/' + stem + '.jpg'
        if os.path.exists(ip):
            shutil.copy(ip, '/kaggle/working/val_sample/'); picked += 1
    if picked >= 8:
        break

!python -m firewatch.cli detect \
    --model "{OUT}" \
    --input /kaggle/working/val_sample \
    --backend tf --arch retinanet \
    --image-size {IMG} --score-thr 0.5 \
    --out /kaggle/working/detect_out

from IPython.display import Image, display
for p in sorted(glob.glob('/kaggle/working/detect_out/*'))[:8]:
    print(p); display(Image(p))